# Stability

The scheme derived on the prior page uses the forward difference scheme for discretizing (and solving) the groundwater flow equations in time. This is an _explicit_ scheme because the solution at time step $n+1$ is based only on "known" information: that of the solution at time step $n$. Unfrotunately, this simplicity comes with a drawback: it is _conditionally stable._

Recall the definition of $r$ from the previous page:

$$
r = \frac{D\,\Delta t}{\Delta x^2}.
$$

This scheme is stable if and only if $r \le 1/2$, which limits the time step to $\Delta t \le \Delta x^2 / (2D)$.

## Demonstration: Case 1 with a chosen value of $r$

The demonstration below repeats the Case 1 example from the prior page, but now $r$ is set directly with a slider instead of being chosen automatically. The time step follows from $r$:

$$
\Delta t = \frac{r\,\Delta x^2}{D}, \qquad N = \frac{t}{\Delta t}.
$$

Because $N$ must be a whole number, the solution is shown at $t_N = N\,\Delta t$, which may differ slightly from the slider value of $t$. The analytical solution is evaluated at the same time.

Try the following:
- Start with $r < 0.5$ and confirm that the numerical solution follows the analytical curve.
- Increase $r$ just past $0.5$. Oscillations appear and grow with each time step.
- With $r > 0.5$, increase $t$ or $M$. More time steps are taken, so the oscillations grow larger.

In [1]:
import sys

if sys.platform == "emscripten":
    import micropip
    await micropip.install(["plotly", "ipywidgets"])

from math import erfc

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
from IPython.display import HTML, display


def update_matrix(M, r):
    """Explicit update matrix for M increments (M + 1 nodes), no boundary conditions."""
    n_nodes = M + 1
    A = (1 - 2 * r) * np.eye(n_nodes)
    A += r * np.eye(n_nodes, k=1)
    A += r * np.eye(n_nodes, k=-1)
    return A


def apply_dirichlet(A):
    """Replace the first and last rows of A with identity rows (fixed boundary values)."""
    A = A.copy()
    A[0, :] = 0.0
    A[0, 0] = 1.0
    A[-1, :] = 0.0
    A[-1, -1] = 1.0
    return A


def case1_analytical(x, t, D, c0):
    return c0 * np.vectorize(erfc)(x / (2 * np.sqrt(D * t)))

In [2]:
def solve_with_r(D, L, M, t, r, c0):
    """Explicit Case 1 solution with a user-chosen r; returns x, c and the time actually reached."""
    x = np.linspace(0, L, M + 1)
    dx = L / M
    dt = r * dx**2 / D
    N = max(1, round(t / dt))

    A = apply_dirichlet(update_matrix(M, r))
    c = np.zeros(M + 1)
    c[0] = c0
    with np.errstate(over="ignore", invalid="ignore"):
        for n in range(N):
            c = A @ c
    return x, c, dx, dt, N, N * dt


def plot_stability(r=0.45, M=40, D=1.0, t=2.0, L=10.0):
    c0 = 1.0
    x_num, c_num, dx, dt, N, t_N = solve_with_r(D, L, M, t, r, c0)
    x_exact = np.linspace(0, L, 500)
    error = np.max(np.abs(c_num - case1_analytical(x_num, t_N, D, c0)))
    stable = r <= 0.5

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_exact, y=case1_analytical(x_exact, t_N, D, c0), mode="lines", name="Analytical"))
    fig.add_trace(go.Scatter(x=x_num, y=c_num, mode="lines+markers", name=f"Finite difference (M = {M})"))
    fig.add_annotation(
        text=(
            f"r = {r:.2f} ({'stable' if stable else 'UNSTABLE'})<br>"
            f"Δx = {dx:.3g}, Δt = {dt:.3g}<br>"
            f"N = {N} time steps, t<sub>N</sub> = {t_N:.3g}<br>"
            f"max |error| = {error:.3g}"
        ),
        xref="paper",
        yref="paper",
        x=0.98,
        y=0.98,
        xanchor="right",
        yanchor="top",
        align="left",
        showarrow=False,
        bordercolor="green" if stable else "red",
        borderwidth=2,
        bgcolor="white",
    )
    fig.update_layout(
        title=f"Stability of the explicit scheme: r = {r:.2f}",
        xaxis_title="Distance from the origin, x",
        yaxis_title="Potential c(x, t)",
        yaxis_range=[-0.5 * c0, 1.5 * c0],
        width=750,
        height=450,
    )
    display(HTML(fig.to_html(include_plotlyjs="cdn")))


widgets.interact(
    plot_stability,
    r=widgets.FloatSlider(value=0.45, min=0.2, max=0.7, step=0.01, description="r"),
    M=widgets.IntSlider(value=40, min=5, max=100, step=1, description="M"),
    D=widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="D"),
    t=widgets.FloatSlider(value=2.0, min=0.1, max=5.0, step=0.1, description="t"),
    L=widgets.FloatSlider(value=10.0, min=5.0, max=20.0, step=1.0, description="L"),
);

interactive(children=(FloatSlider(value=0.45, description='r', max=0.7, min=0.2, step=0.01), IntSlider(value=4…